In [5]:
import pandas as pd

# Read the CSV file
df = pd.read_csv('logs/ravdess-speech_attribute_SAIL-predictions.csv')

#Drop all entries that start with the name 03-01-01, since they're neutral and have no intensity variation.
df = df[~df['Filename'].str.startswith('03-01-01')]

#Add a new column with emotion category based on the filename
def get_emotion_category(filename):
    emotion_code = filename.split('-')[2]
    emotion_map = {
        '01': 'neutral',
        '02': 'calm',
        '03': 'happy',
        '04': 'sad',
        '05': 'angry',
        '06': 'fearful',
        '07': 'disgust',
        '08': 'surprised'
    }
    return emotion_map.get(emotion_code, 'unknown')

def get_emotion_intensity(filename):
    intensity_code = filename.split('-')[3]
    intensity_map = {
        '01': 'normal',
        '02': 'strong'
    }
    return intensity_map.get(intensity_code, 'unknown')

df['EmoClass'] = df['Filename'].apply(get_emotion_category)
df['Intensity'] = df['Filename'].apply(get_emotion_intensity)
df['SpkrID'] = df['Filename'].apply(lambda x: x.split('-')[6].split('.')[0])
df['Take'] = df['Filename'].apply(lambda x: x.split('-')[5])
df['Sentence'] = df['Filename'].apply(lambda x: x.split('-')[4])

# Rename columns to match MSP-Podcast format
df.rename(columns={'Arousal': 'EmoAct'}, inplace=True)
df.rename(columns={'Valence': 'EmoVal'}, inplace=True)
df.rename(columns={'Dominance': 'EmoDom'}, inplace=True)
df.rename(columns={'FileName': 'EmoDom'}, inplace=True)

#Reorder rows by speaker, then category, then intensity, then repetition number
df = df.sort_values(by=['SpkrID', 'EmoClass', 'Sentence', 'Intensity', 'Take'])

df.to_csv('logs/ravdess_attribute_differences.csv', index=False)
